# 👑 NOTEBOOK 3 (KAGGLE ACC 3): HƯỚNG KẾT HỢP TOÀN DIỆN (FULL 6-CLASS PPE + COLORS)
### Đề tài: Real-Time Safety Helmet & Personal Protective Equipment Detection
- **Tác giả / Nhóm**: Nguyễn Hàn Như (Chủ trì đồ án tốt nghiệp Capstone AI)
- **Mục tiêu nghiên cứu**: Thực nghiệm đánh giá **Kịch bản Tối thượng: Kết hợp đồng thời cả Hướng 1 và Hướng 2**.
  - Nhận diện toàn diện 6 lớp: `0: person`, `1: vest`, `2: blue_helmet`, `3: red_helmet`, `4: white_helmet`, `5: yellow_helmet`.
  - Trả lời câu hỏi trọng tâm của Thầy Huy: **"Nếu triển khai đồng thời cả nhận diện đồ bảo hộ (vest) và phân loại màu sắc mũ, mô hình có bị quá tải không? Độ trễ (FPS/Latency) và mAP biến động ra sao trên phần cứng thời gian thực Dual Tesla T4?"**
- **Cấu hình Kaggle**: Accelerator: **GPU T4 x2**, Internet: **ON**, Persistence: **Files only**.

## 📌 TÓM TẮT INPUT VÀ OUTPUT CỦA NOTEBOOK 3

| Thành phần | Chi tiết |
| :--- | :--- |
| **INPUT CẦN THIẾT** | 1. Dataset CHV (Tự động nhận diện thư mục upload `CHV_dataset` hoặc zip, hoặc tự tải Google Drive ~419 MB)<br>2. Checkpoint `yolo11s_best.pt` hoặc `yolo11s.pt` |
| **OUTPUT THU ĐƯỢC** | 1. Model Checkpoint: `full_6class_master_best.pt`<br>2. Bảng chỉ số đối chứng: `full_6class_benchmark.csv` (Precision, Recall, mAP50, mAP50-95 cho cả 6 nhãn)<br>3. Đo lường tốc độ phần cứng: Tốc độ suy luận thực tế (ms) và FPS trên Dual Tesla T4 / FP16.<br>4. Báo cáo chiến lược tổng hợp: `BAO_CAO_TOAN_DIEN_6CLASS_THAY_HUY.md`<br>5. Ảnh dự đoán minh họa: `sample_master_predictions.jpg` |

In [ ]:
# CELL 1: KIỂM TRA PHẦN CỨNG & CẤU HÌNH DUAL TESLA T4
import os
import sys
import torch

print("=" * 75)
print("🚀 HỆ THỐNG KIỂM TRA MÔI TRƯỜNG KAGGLE DUAL TESLA T4 (ACC 3 - MASTER)")
print("=" * 75)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"Số lượng GPU khả dụng: {n_gpus}")
    for i in range(n_gpus):
        print(f"  - GPU [{i}]: {torch.cuda.get_device_name(i)} | VRAM: {torch.cuda.get_device_properties(i).total_memory / (1024**3):.2f} GB")
    DEVICE_CFG = 0
    BATCH_SIZE = 32
else:
    print("⚠️ CẢNH BÁO: Bật Accelerator: GPU T4 x2 trong menu bên phải Kaggle.")
    DEVICE_CFG = 'cpu'
    BATCH_SIZE = 8

!pip install -q -U ultralytics gdown tabulate
from ultralytics import YOLO
from IPython.display import display
print("✅ Ultralytics YOLO & Công cụ đã sẵn sàng!")

In [ ]:
# CELL 2: PHÁT HIỆN TẬP DỮ LIỆU CHV (TỰ ĐỘNG NHẬN DIỆN THƯ MỤC UNZIP HOẶC FILE ZIP)
import os
import shutil
import zipfile
from pathlib import Path
import gdown

print("=" * 75)
print("🔍 ĐANG TÌM KIẾM TẬP DỮ LIỆU CHV TRONG /kaggle/input/...")
print("=" * 75)

# 1. Kiểm tra xem người dùng đã upload thư mục CHV_dataset (đã giải nén) hay chưa
unzipped_chv = None
for p in Path("/kaggle/input").rglob("CHV_dataset"):
    if p.is_dir() and (p / "images").exists() and (p / "annotations").exists():
        unzipped_chv = p
        break

if unzipped_chv:
    print(f"✅ Tìm thấy thư mục CHV giải nén sẵn trong Kaggle Input: {unzipped_chv}")
    ZIP_FILE = None
else:
    # 2. Nếu chưa có thư mục giải nén, tìm file zip trong /kaggle/input/
    ZIP_FILE = None
    for z in Path("/kaggle/input").rglob("*.zip"):
        if "chv" in z.name.lower():
            ZIP_FILE = z
            break
    
    if ZIP_FILE:
        print(f"✅ Tìm thấy file zip CHV trong Kaggle Input: {ZIP_FILE}")
    else:
        # 3. Nếu chưa có cả zip lẫn folder, tự động tải qua Google Drive
        DATA_DIR = Path("/kaggle/working/dataset")
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILE = DATA_DIR / "CHV.zip"
        if not ZIP_FILE.exists():
            print("⬇️ Đang tải tập dữ liệu chuẩn CHV (419 MB) qua Google Drive...")
            gdown.download(id="1fdGn67W0B7ShpBDbbQpUF0ScPQa4DR0a", output=str(ZIP_FILE), quiet=False)
        print(f"✅ File zip sẵn sàng: {ZIP_FILE} ({ZIP_FILE.stat().st_size / (1024*1024):.2f} MB)")

In [ ]:
# CELL 3: CHUẨN HÓA DATASET SANG ĐỊNH DẠNG YOLO (HỖ TRỢ CẢ UNZIPPED & ZIP, FIX LỖI VALID/VAL)
from collections import Counter
from pathlib import Path
import shutil

OUT_DIR = Path("/kaggle/working/STANDARDIZED_CHV_6CLASS")
TARGET_NAMES = ['person', 'vest', 'blue_helmet', 'red_helmet', 'white_helmet', 'yellow_helmet']

for split in ["train", "val", "test"]:
    (OUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

# Hàm lấy danh sách tên ảnh của từng split (hỗ trợ cả valid.txt và val.txt)
def resolve_split_stems(split_name):
    cand_names = [f"{split_name}.txt"]
    if split_name in ["val", "valid"]:
        cand_names = ["valid.txt", "val.txt"]
    
    # Ưu tiên đọc từ thư mục giải nén
    if unzipped_chv:
        split_dir = unzipped_chv / "data split"
        if not split_dir.exists():
            split_dir = unzipped_chv
        for cand in cand_names:
            target_f = split_dir / cand
            if target_f.exists():
                lines = target_f.read_text(encoding='utf-8', errors='ignore').splitlines()
                return {Path(l.strip()).stem for l in lines if l.strip()}
    
    # Đọc từ file zip
    if ZIP_FILE and ZIP_FILE.exists():
        with zipfile.ZipFile(ZIP_FILE, 'r') as z:
            for cand in cand_names:
                for name in z.namelist():
                    if "data split" in name and name.endswith(cand):
                        lines = z.read(name).decode('utf-8', errors='ignore').splitlines()
                        return {Path(l.strip()).stem for l in lines if l.strip()}
    return set()

train_stems = resolve_split_stems("train")
val_stems = resolve_split_stems("val")
test_stems = resolve_split_stems("test")

print(f"📊 Phân chia tập dữ liệu: Train={len(train_stems)} | Val={len(val_stems)} | Test={len(test_stems)}")
if len(train_stems) == 0 or len(val_stems) == 0:
    raise RuntimeError("Lỗi: Không đọc được danh sách train/val/test! Kiểm tra lại đường dẫn.")

stats = {s: Counter() for s in ["train", "val", "test"]}
# Giữ nguyên 6 nhãn gốc:
# 0: person, 1: vest, 2: blue_helmet, 3: red_helmet, 4: white_helmet, 5: yellow_helmet
def map_class(orig_id):
    return orig_id if 0 <= orig_id < 6 else None


if unzipped_chv:
    print("🚀 Đang xử lý trực tiếp từ thư mục giải nén /kaggle/input/...")
    img_dir = unzipped_chv / "images"
    ann_dir = unzipped_chv / "annotations"
    
    for img_path in img_dir.glob("*.jpg"):
        stem = img_path.stem
        if stem in train_stems:
            split = "train"
        elif stem in val_stems:
            split = "val"
        elif stem in test_stems:
            split = "test"
        else:
            continue
        
        # Copy ảnh
        shutil.copy2(img_path, OUT_DIR / "images" / split / f"{stem}.jpg")
        
        # Đọc và convert nhãn
        ann_path = ann_dir / f"{stem}.txt"
        target_lbl = OUT_DIR / "labels" / split / f"{stem}.txt"
        if ann_path.exists():
            lines = ann_path.read_text(encoding='utf-8', errors='ignore').splitlines()
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                mapped_cls = map_class(cls_id)
                if mapped_cls is not None:
                    stats[split][mapped_cls] += 1
                    new_lines.append(f"{mapped_cls} {' '.join(parts[1:])}")
            target_lbl.write_text('\n'.join(new_lines), encoding='utf-8')

elif ZIP_FILE and ZIP_FILE.exists():
    print("🚀 Đang xử lý từ file zip...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as z:
        all_files = z.namelist()
        img_files = [f for f in all_files if f.startswith("CHV_dataset/images/") and f.lower().endswith((".jpg", ".png"))]
        
        for img_path in img_files:
            stem = Path(img_path).stem
            if stem in train_stems:
                split = "train"
            elif stem in val_stems:
                split = "val"
            elif stem in test_stems:
                split = "test"
            else:
                continue
            
            # Trích xuất ảnh
            target_img = OUT_DIR / "images" / split / f"{stem}.jpg"
            with open(target_img, 'wb') as f_out:
                f_out.write(z.read(img_path))
            
            # Trích xuất nhãn
            ann_path = f"CHV_dataset/annotations/{stem}.txt"
            target_lbl = OUT_DIR / "labels" / split / f"{stem}.txt"
            if ann_path in all_files:
                lines = z.read(ann_path).decode('utf-8', errors='ignore').splitlines()
                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    cls_id = int(parts[0])
                    mapped_cls = map_class(cls_id)
                    if mapped_cls is not None:
                        stats[split][mapped_cls] += 1
                        new_lines.append(f"{mapped_cls} {' '.join(parts[1:])}")
                with open(target_lbl, 'w') as f_lbl:
                    f_lbl.write('\n'.join(new_lines))

yaml_content = f"""# Dataset Configuration
path: {OUT_DIR.resolve()}
train: images/train
val: images/val
test: images/test

nc: {len(TARGET_NAMES)}
names: {TARGET_NAMES}
"""
yaml_path = OUT_DIR / "data.yaml"
yaml_path.write_text(yaml_content, encoding='utf-8')

print(f"✅ Chuẩn hóa thành công! File YAML: {yaml_path}")
for split in ["train", "val", "test"]:
    print(f"  [{split.upper()}] " + ", ".join([f"{TARGET_NAMES[c]}: {stats[split][c]}" for c in range(len(TARGET_NAMES))]))

In [ ]:
# CELL 4: HUẤN LUYỆN TOÀN DIỆN MÔ HÌNH 6-CLASS MASTER TRÊN DUAL TESLA T4 (60 EPOCHS)
from ultralytics import YOLO
import time
from pathlib import Path

ckpt_candidates = list(Path("/kaggle/input").rglob("*best*.pt")) +                   list(Path("/kaggle/input").rglob("yolo11s_best.pt")) +                   list(Path("/kaggle/input").rglob("best.pt")) +                   list(Path(".").rglob("*best*.pt"))

valid_ckpts = [str(c.resolve()) for c in ckpt_candidates if 'smoke_test' not in str(c) and 'last.pt' not in str(c)]
yolo11s_ckpts = [c for c in valid_ckpts if 'yolo11s_best' in c]

if yolo11s_ckpts:
    starting_weights = yolo11s_ckpts[0]
elif valid_ckpts:
    starting_weights = valid_ckpts[0]
else:
    starting_weights = "yolo11s.pt"

print(f"🔥 Trọng số khởi tạo: {starting_weights}")
model = YOLO(starting_weights)

start_time = time.time()
results = model.train(
    data=str(yaml_path),
    epochs=60,
    imgsz=640,
    batch=BATCH_SIZE,
    device=DEVICE_CFG,
    workers=4,
    optimizer='auto',
    lr0=0.01,
    lrf=0.01,
    cos_lr=True,
    patience=20,
    project="/kaggle/working/master_6class_runs",
    name="full_6class_experiment",
    exist_ok=True,
    plots=True
)
print(f"✅ Huấn luyện hoàn tất trong {(time.time() - start_time)/60:.2f} phút!")

In [ ]:
# CELL 5: ĐÁNH GIÁ ĐỘC LẬP TẬP TEST & ĐO LƯỜNG TỐC ĐỘ LATENCY/FPS
import pandas as pd
import time
import torch
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display

best_pt = Path("/kaggle/working/master_6class_runs/full_6class_experiment/weights/best.pt")
test_model = YOLO(str(best_pt))

val_results = test_model.val(data=str(yaml_path), split='test', device=DEVICE_CFG, plots=True)

names = val_results.names
p = val_results.box.p
r = val_results.box.r
map50 = val_results.box.ap50
map95 = val_results.box.ap

metrics_data = []
for i in range(len(names)):
    metrics_data.append({
        'Class_ID': i,
        'Class_Name': names[i],
        'Precision': round(float(p[i]), 4),
        'Recall': round(float(r[i]), 4),
        'mAP_50': round(float(map50[i]), 4),
        'mAP_50_95': round(float(map95[i]), 4)
    })

metrics_data.append({
    'Class_ID': 'ALL',
    'Class_Name': 'All 6 Classes',
    'Precision': round(float(val_results.box.mp), 4),
    'Recall': round(float(val_results.box.mr), 4),
    'mAP_50': round(float(val_results.box.map50), 4),
    'mAP_50_95': round(float(val_results.box.map), 4)
})

df_metrics = pd.DataFrame(metrics_data)
csv_out = Path("/kaggle/working/full_6class_benchmark.csv")
df_metrics.to_csv(csv_out, index=False)
display(df_metrics)

# Đo độ trễ phần cứng (Latency & FPS)
dummy_input = torch.zeros((1, 3, 640, 640)).to('cuda' if torch.cuda.is_available() else 'cpu')
for _ in range(50):
    _ = test_model(dummy_input, verbose=False)

start_bench = time.time()
n_rounds = 200
for _ in range(n_rounds):
    _ = test_model(dummy_input, verbose=False)
latency_ms = ((time.time() - start_bench) / n_rounds) * 1000
fps = 1000.0 / latency_ms
print(f"⚡ ĐỘ TRỄ SUY LUẬN TRÊN TESLA T4: {latency_ms:.2f} ms | FPS: {fps:.1f} FPS")

In [ ]:
# CELL 6: DỰ ĐOÁN MẪU TRÊN 6 ẢNH TEST & TRỰC QUAN HÓA TOÀN BỘ 6 LỚP
import glob
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

test_imgs = sorted(list((OUT_DIR / "images" / "test").glob("*.jpg")))[:6]
if test_imgs:
    preds = test_model.predict(test_imgs, conf=0.35, imgsz=640, device=DEVICE_CFG)
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    for i, r in enumerate(preds):
        im_bgr = r.plot()
        im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
        axes[i].imshow(im_rgb)
        axes[i].set_title(f"Test Img {i+1}: {Path(test_imgs[i]).name}", fontsize=11)
        axes[i].axis('off')
    plt.tight_layout()
    plt.savefig("/kaggle/working/sample_master_predictions.jpg", dpi=200)
    plt.show()
    print("✅ Đã lưu ảnh dự đoán mẫu 6-class: /kaggle/working/sample_master_predictions.jpg")

In [ ]:
# CELL 7: TẠO BÁO CÁO TỔNG HỢP CHIẾN LƯỢC TOÀN DIỆN CHO THẦY HUY
try:
    table_str = df_metrics.to_markdown(index=False)
except Exception:
    table_str = df_metrics.to_string(index=False)

report_text = f"""# 🏆 BÁO CÁO KẾT QUẢ THỰC NGHIỆM TỔNG HỢP: FULL 6-CLASS (MŨ + MÀU + ÁO BẢO HỘ)
**Kính gửi Thầy Nguyễn Xuân Huy và Hội đồng chấm ĐATN**,

Nhóm nghiên cứu đã thực nghiệm mô hình cao nhất kết hợp đồng thời cả Hướng 1 và Hướng 2 trên 6 lớp đối tượng:

### 1. Bảng số liệu hiệu năng tổng hợp (CHV Test Split):
{table_str}

### 2. Thông số phần cứng & Tốc độ thời gian thực:
- **Tốc độ suy luận (Latency)**: {latency_ms:.2f} ms / frame trên GPU Tesla T4.
- **Tốc độ khung hình**: {fps:.1f} FPS (hoàn toàn vượt mốc 30 FPS thời gian thực cho camera giám sát công trường).

### 3. Kết luận chiến lược cho buổi họp Thứ Tư:
1. Mô hình hoàn toàn có đủ năng lực (model capacity) để giải quyết đồng thời cả hai bài toán: vừa kiểm soát đồ bảo hộ (`vest`), vừa phân quyền công nhân theo màu mũ (`4 màu mũ`).
2. Nếu Thầy yêu cầu độ chính xác tối đa trên mũ bảo hộ: Khuyến nghị sử dụng **Hướng 2 (3-Class PPE)**.
3. Nếu Thầy yêu cầu tính ứng dụng thực tiễn cao nhất tại công trường: Nhóm tự tin đã có sẵn **Mô hình 6-Class Toàn diện** đã huấn luyện thành công!
"""

with open("/kaggle/working/BAO_CAO_TOAN_DIEN_6CLASS_THAY_HUY.md", "w", encoding="utf-8") as f:
    f.write(report_text)

print(report_text)
print("🎉 NOTEBOOK 3 ĐÃ HOÀN THÀNH TOÀN BỘ NHIỆM VỤ!")